In [46]:
import numpy as np
import matplotlib.pyplot as plt
import csv

In [50]:
def generate_random_line_params():
    a = np.random.uniform(0.5, 3.0)
    b = np.random.uniform(-5.0, 10.0)
    return a, b

def generate_noise_params():
    noise_std = np.random.uniform(7, 25)
    return noise_std

def generate_main_data(a, b, noise_std, N=1000, x_range=(0, 100)):
    x_main = np.linspace(x_range[0] + 15, x_range[1] - 15, N)
    y_clean = a * x_main + b
    noise = np.random.normal(0, noise_std, N)
    y_noisy = y_clean + noise
    return x_main, y_clean, y_noisy

def generate_random_outlier_params(a, b, n_out, k, x_range=(0,100)):
    num_outliers = int(((50*k)/(1-0.05*k))/n_out)
    
    # Выбираем случайное значение x в заданном диапазоне
    outlier_center_x = np.random.uniform(x_range[0] + 10, x_range[1] - 10)
    # Вычисляем истинное значение y для выбранного x и добавляем случайное отклонение
    true_y = a * outlier_center_x + b
    outlier_center_y = true_y + np.random.uniform(50, 100) * np.random.choice([-1, 1])
    outlier_std = np.random.uniform(2, 7)
    
    return num_outliers, outlier_center_x, outlier_center_y, outlier_std

def generate_outliers(num_outliers, center_x, center_y, std):
    x_out = np.random.normal(center_x, std, num_outliers)
    y_out = np.random.normal(center_y, std, num_outliers)
    return x_out, y_out

def save_data_to_csv(X, Y, filename):
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['x', 'y'])
        for xi, yi in zip(X, Y):
            writer.writerow([xi, yi])

def generate_linear_outlier_line_params(x_range=(0,100)):
    a_out = np.random.uniform(-3.0, 3.0)       
    b_out = np.random.uniform(-20.0, 20.0)        
    return a_out, b_out

def generate_linear_outliers(num_points, a_out, b_out, x_range=(0,100), noise_std=2.0):
    x_out = np.linspace(x_range[0] + 5, x_range[1] - 5, num_points)
    y_line = a_out * x_out + b_out
    
    y_out = y_line + np.random.normal(0, noise_std, num_points)
    return x_out, y_out

def generate_random_noise_outliers(num_points, x_range, y_range):
    x_out = np.random.uniform(x_range[0], x_range[1], num_points)
    y_out = np.random.uniform(y_range[0], y_range[1], num_points)
    return x_out, y_out


In [ ]:
def gen_outlier(n_out, k, category = "stat"):
  # Генерация случайных параметров для линии и шума
  a, b = generate_random_line_params()
  noise_std = generate_noise_params()

  # Генерация основной выборки
  N = 1000
  x_range = (0, 100)
  x_main, y_clean, y_noisy = generate_main_data(a, b, noise_std, N, x_range)

  # Генерация случайных параметров для выбросов и их создание
  filename_base = f"{a:.2f}_{b:.2f}"

  outlier_x_list = []
  outlier_y_list = []


  if (category == "stat"):
    filename_base += f"_s"
    for _ in range(n_out):
      num_outliers, outlier_center_x, outlier_center_y, outlier_std = generate_random_outlier_params(a, b, n_out, k, x_range)
      x_outliers, y_outliers = generate_outliers(num_outliers, outlier_center_x, outlier_center_y, outlier_std)
      outlier_x_list.append(x_outliers)
      outlier_y_list.append(y_outliers)
      filename_base += f"_{num_outliers}"
  elif (category == "lin"):
    filename_base += f"_l"
    for _ in range(n_out):
      num_points = int(((50*k)/(1-0.05*k)) / n_out)
      a_out, b_out = generate_linear_outlier_line_params(x_range)

      x_out, y_out = generate_linear_outliers(num_points, a_out, b_out, x_range, noise_std)
      outlier_x_list.append(x_out)
      outlier_y_list.append(y_out)
      
      filename_base += f"_{num_points}"
  else:
    filename_base += f"_r"

    x_range = (0, 100)
    y_max = np.max(y_clean)
    y_range = (-25, y_max + 25)
    num_points = int(((50*k)/(1-0.05*k)) / n_out)

    x_out, y_out = generate_random_noise_outliers(num_points, x_range, y_range)
    outlier_x_list.append(x_out)
    outlier_y_list.append(y_out)

    filename_base += f"_{num_points}"
    
    
  # Объединяем основную выборку и все выбросы
  X = np.concatenate((x_main, *outlier_x_list))
  Y = np.concatenate((y_noisy, *outlier_y_list))

  csv_filename = f"data_outliers/{filename_base}.csv"
  plot_filename = f"graph_outliers/{filename_base}.png"

  # Сохраняем данные в CSV
  save_data_to_csv(X, Y, csv_filename)

  # Визуализация результата
  plt.figure(figsize=(8, 6))
  plt.scatter(X, Y, color='gray', alpha=0.7, label='Данные с выбросами')
  plt.plot(x_main, y_clean, color='red', linewidth=2, label='Истинная линия')
  plt.title('Линейная зависимость с аддитивным шумом и выбросами')
  plt.xlabel('x')
  plt.ylabel('y')
  plt.legend()
  plt.grid(True)

  # Сохраняем график
  plt.savefig(plot_filename)
  plt.show()

# for i in range (1,6):
#   for k in range(14):
#     gen_outlier(i,k, "stat")

# for i in range (1,3):
#   for k in range(14):
#     gen_outlier(i,k, "lin")

for _ in range (1,3):
  for k in range(14):
    gen_outlier(1,k, "rand")
